1. **Load**

In [50]:
import pandas as pd
import numpy as np

from pathlib import Path

import spacy
from negspacy.negation import Negex

fyp_dir = Path.cwd().parent
file_path = fyp_dir / "data"

train_df = pd.read_csv(f"{file_path}/train_split.csv")
test_df  = pd.read_csv(f"{file_path}/test_split.csv")

train_df.head()

,subject_id,stay_id,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,acuity,chiefcomplaint,risk,text_len_words
0,13561687,30620178,98.6,79.0,18.0,99.0,123.0,62.0,0,3.0,"Fever, Nausea",Medium,2
1,18510804,36501273,96.0,89.0,22.0,98.0,96.0,68.0,5,3.0,"Asthma exacerbation, Cough, Headache",Medium,4
2,10012292,36122045,99.0,111.0,28.0,80.0,139.0,76.0,NaN,1.0,HYPOXIA,High,1
3,11554437,39823841,97.1,117.0,18.0,100.0,116.0,61.0,8,3.0,SOB,Medium,1
4,14814421,31009844,98.1,73.0,16.0,100.0,159.0,98.0,0,2.0,"SI, Palpitations",High,2


2. **Cleansing**

In [51]:
import re

ABBR_MAP = {
    "cp": "chest pain",
    "sob": "shortness of breath",
    "s.o.b": "shortness of breath",
    "c/p": "chest pain",
    "soa": "shortness of breath",
    "n/v": "nausea vomiting",
    "n/vom": "nausea vomiting",
    "n/vomiting": "nausea vomiting",
    "abd pain": "abdominal pain",
    "ap": "abdominal pain",
    "ha": "headache",
    "h/a": "headache",
    "diz": "dizziness",
    "loc": "loss of consciousness",
    "syn": "syncope",
    "faint": "syncope",
    "uti": "urinary tract infection",
    "uti.": "urinary tract infection",
    "uti/s": "urinary tract infection",
    "uti?": "urinary tract infection",
    "ms": "medical complaint",  # example; remove if not relevant
    "hematuria": "hematuria",
    # add what you actually see in your data
}

def normalize_abbr(text: str) -> str:
    t = text.lower()
    # normalize punctuation variants a bit
    t = t.replace(",", " ").replace(";", " ").replace("(", " ").replace(")", " ")
    # replace multiword abbreviations first
    for k, v in sorted(ABBR_MAP.items(), key=lambda x: -len(x[0])):
        # word-boundary-ish replace: avoid replacing inside other words
        t = re.sub(rf"(?<!\w){re.escape(k)}(?!\w)", v, t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def clean_text(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    x = normalize_abbr(x)
    return x
    

train_df["text"] = train_df["chiefcomplaint"].apply(clean_text)
test_df["text"]  = test_df["chiefcomplaint"].apply(clean_text)

3. **Build a negation-aware spaCy pipeline**


a.) Create a spaCy component for Negex

In [55]:
from negspacy.negation import Negex
from spacy.matcher import PhraseMatcher
from spacy.tokens import Span
from spacy.language import Language

nlp = spacy.load("en_core_web_sm")

SYMPTOM_KEYWORDS = [
    "chest pain",
    "shortness of breath",
    "sob",
    "dyspnea",
    "cough",
    "fever",
    "vomiting",
    "nausea",
    "abdominal pain",
    "back pain",
    "headache",
    "dizziness",
    "syncope",
    "faint",
    "weakness",
    "numbness",
    "stroke",
    "bleeding",
    "blood in stool",
    "hematemesis",
    "suicidal",
    "overdose",
    "burn",
    "urinary tract infection",
    "dysuria",
    "hematuria",
    "seizure",
    "neck pain",
    "palpitations",
    "diarrhea",
    "dehydration",
]

@Language.component("keyword_adder")
def keyword_adder(doc):
    matcher = PhraseMatcher(doc.vocab, attr="LOWER")
    patterns = [nlp.make_doc(k) for k in SYMPTOM_KEYWORDS]
    matcher.add("SYMPTOM", patterns)

    matches = matcher(doc)
    spans = [Span(doc, start, end, label="SYMPTOM") for _, start, end in matches]

    # remove overlaps: keep longest spans (simple heuristic)
    spans = sorted(spans, key=lambda s: (s.start_char, -(s.end - s.start)))
    non_overlapping = []
    occupied = set()
    for sp in spans:
        token_ids = set(range(sp.start, sp.end))
        if occupied.isdisjoint(token_ids):
            non_overlapping.append(sp)
            occupied |= token_ids

    doc.ents = non_overlapping
    return doc

if "keyword_adder" not in nlp.pipe_names:
    nlp.add_pipe("keyword_adder", last=True)

if "negex" not in nlp.pipe_names:
    nlp.add_pipe(
        "negex", 
        last=True, 
        config={"ent_types": ["SYMPTOM"]}
    )

4. ** Create a minimal “evidence object” column (for v1) **

In [61]:
def extract_evidence(doc):
    """
    Returns list of dicts:
      {
        "concept": <keyword or matched phrase in lowercase>,
        "negated": bool,
        "span_text": <surface form>,
        "char_start": int,
        "char_end": int
      }
    """
    evidence = []
    for ent in doc.ents:
        neg_flag = None

        # Prefer the attribute that worked for you: ent._.negex
        if hasattr(ent._, "negex"):
            neg_flag = bool(ent._.negex)
        elif hasattr(ent._, "negated"):
            neg_flag = bool(ent._.negated)

        # If neither exists, default to False (but then you should verify pipeline)
        if neg_flag is None:
            neg_flag = False

        evidence.append({
            "concept": ent.text.lower(),
            "negated": neg_flag,
            "span_text": ent.text,
            "char_start": ent.start_char,
            "char_end": ent.end_char,
        })
    return evidence


def preprocess_single(text: str):
    text = clean_text(text)
    doc = nlp(text)

    evidence = extract_evidence(doc)

    return {
        "text": text,
        "evidence": evidence,
        "affirmed_concepts": [e["concept"] for e in evidence if not e["negated"]],
        "negated_concepts":  [e["concept"] for e in evidence if e["negated"]],
    }

In [62]:
doc = nlp("patient denies chest pain and shortness of breath no fever.")
for ent in doc.ents:
    print(ent.text, ent._.negex)  # should work

chest pain True
shortness of breath True
fever True
